# Промпт #31 — ReAct (Reasoning + Acting)

**Техника:** ReAct pattern  
**Задача:** Модель думает и вызывает инструменты — калькулятор и конвертер валют  
**Сложность:** ⭐⭐⭐⭐☆

In [1]:
import sys
sys.path.append('..')
from config import get_completion_messages

In [7]:
# Инструменты которые может вызвать модель
def calculator(expression):
    try:
        return str(eval(expression))
    except:
        return "Ошибка вычисления"

def currency_converter(amount, from_currency, to_currency):
    rates = {"USD_RUB": 89.5, "EUR_RUB": 97.2, "RUB_USD": 0.011}
    key = f"{from_currency}_{to_currency}"
    if key in rates:
        return str(round(amount * rates[key], 2))
    return "Курс не найден"

# System prompt объясняет модели как использовать инструменты
system = """Ты ассистент который решает задачи используя инструменты.

Правила:
1. Пиши ТОЛЬКО ONE Thought и ONE Action за раз
2. После Action сразу СТОП — не пиши больше ничего
3. Жди Observation перед следующим шагом
4. Никогда не выдумывай результаты инструментов

Формат:
Thought: [одна мысль]
Action: [один вызов инструмента]

Доступные инструменты:
- calculator(выражение)
- currency_converter(сумма, из_валюты, в_валюту)

Когда есть все данные для финального ответа:
Answer: [ответ]"""

task = "У меня есть 500 долларов и 300 евро. Сколько это всего в рублях?"
messages = [{"role": "user", "content": task}]
print(f"Задача: {task}\n")

# ReAct цикл — максимум 6 итераций
for i in range(6):
    response = get_completion_messages(messages, system_prompt=system, temperature=0.1)
    messages.append({"role": "assistant", "content": response})
    print(f"=== ШАГ {i+1} ===")
    print(response)

    # Если модель дала финальный ответ — стоп
    if "Answer:" in response:
        break

    # Парсим Action и запускаем инструмент
    if "Action:" in response:
        action_line = [l for l in response.split("\n") if "Action:" in l][0]
        action = action_line.replace("Action:", "").strip()

        # Определяем какой инструмент вызвать
        if action.startswith("calculator"):
            expr = action.replace("calculator(", "").rstrip(")")
            result = calculator(expr)
        elif action.startswith("currency_converter"):
            params = action.replace("currency_converter(", "").rstrip(")")
            parts = [p.strip() for p in params.split(",")]
            result = currency_converter(float(parts[0]), parts[1], parts[2])
        else:
            result = "Инструмент не найден"

        # Передаём результат обратно модели
        observation = f"Observation: {result}"
        messages.append({"role": "user", "content": observation})
        print(f"{observation}\n")

Задача: У меня есть 500 долларов и 300 евро. Сколько это всего в рублях?

=== ШАГ 1 ===
Thought: Чтобы узнать общую сумму в рублях, мне нужно сначала перевести обе суммы в рубли, для этого мне нужно знать текущий курс валют, но сначала я могу перевести доллары в рубли или евро в рубли, но мне нужно знать курс валют, поэтому я начну с перевода евро в доллары, чтобы иметь одну и ту же валюту.

Action: currency_converter(300, "EUR", "USD")
Observation: Курс не найден

=== ШАГ 2 ===
Thought: Поскольку курс валют не найден, мне нужно найти другой способ решения задачи, например, найти курс валют вручную или использовать другой инструмент, но в данном случае я не имею доступа к курсу валют, поэтому мне нужно изменить подход и попробовать найти курс доллар к рублю и евро к рублю отдельно.

Action: calculator(1)
Observation: 1

=== ШАГ 3 ===
Thought: Результат от калькулятора не имеет отношения к задаче, мне нужно найти курс валют, но у меня нет доступа к курсу валют, поэтому мне нужно попробо

## Оценка: 5/5

## Инсайт
Идеальный edge case — модель столкнулась с отсутствующим курсом
EUR→USD и начала деградировать: calculator(1), Action: None,
и в итоге отказалась решать задачу.

Что пошло не так:
- Модель выбрала неоптимальный путь (EUR→USD вместо EUR→RUB)
- Получила "Курс не найден" и потеряла ориентир
- Не смогла переключиться на альтернативный путь

Это показывает главный риск ReAct: если один шаг возвращает
неожиданный результат — модель может зациклиться или сдаться.

Как исправить в продакшене:
1. Добавить fallback в инструменты — возвращать подсказку
   вместо "Курс не найден": "Курс EUR_USD недоступен.
   Попробуй EUR_RUB или USD_RUB"
2. Добавить в system prompt примеры обработки ошибок
3. Использовать нативный tool use API который более надёжен

Главный вывод: ReAct хрупок на ошибках инструментов.
Надёжность системы = н